## Module 2 | DAY 4: Optimization foundations

### 1. Loss (Cost) Functions

#### 1.1 Introduction to Loss Functions

A loss function measures how far a model's predictions are from the actual values. The goal of training is to minimize this loss.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Common loss functions
def mse(y_true, y_pred):
    """Mean Squared Error"""
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def huber_loss(y_true, y_pred, delta=1.0):
    """Huber Loss - combines MSE and MAE"""
    diff = y_true - y_pred
    is_small = np.abs(diff) <= delta
    squared_loss = 0.5 * diff ** 2
    linear_loss = delta * (np.abs(diff) - 0.5 * delta)
    return np.where(is_small, squared_loss, linear_loss)

# Generate data
y_true = np.array([1, 2, 3, 4, 5])
y_pred = np.array([1.2, 1.8, 3.5, 3.8, 5.5])

print("Loss Functions Comparison:")
print(f"True values: {y_true}")
print(f"Predictions: {y_pred}")
print(f"\nMSE: {mse(y_true, y_pred):.4f}")
print(f"MAE: {mae(y_true, y_pred):.4f}")
print(f"Huber Loss: {np.mean(huber_loss(y_true, y_pred)):.4f}")

# Visualize different loss functions
plt.figure(figsize=(12, 4))

# Plot 1: Loss values for different predictions
errors = np.linspace(-3, 3, 100)
mse_values = errors ** 2
mae_values = np.abs(errors)
huber_values = huber_loss(0, errors, delta=1.0)

plt.subplot(1, 2, 1)
plt.plot(errors, mse_values, 'b-', label='MSE', linewidth=2)
plt.plot(errors, mae_values, 'r-', label='MAE', linewidth=2)
plt.plot(errors, huber_values, 'g-', label='Huber', linewidth=2)
plt.xlabel('Error (y_true - y_pred)')
plt.ylabel('Loss Value')
plt.title('Loss Function Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Model predictions visualization
plt.subplot(1, 2, 2)
plt.scatter(y_true, y_pred, color='blue', s=100, alpha=0.6)
plt.plot([0, 6], [0, 6], 'r--', label='Perfect Prediction')
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('Predictions vs Reality')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 6)
plt.ylim(0, 6)

plt.tight_layout()
plt.show()

print("\n💡 ML Connection:")
print("MSE is sensitive to outliers (penalizes large errors heavily)")
print("MAE is more robust to outliers")
print("Huber combines advantages of both")

### 1.2 Classification Loss Functions

Cross-entropy loss is commonly used for classification problems.

In [ ]:
# Binary Cross-Entropy Loss
def binary_cross_entropy(y_true, y_pred):
    """Binary Cross-Entropy Loss"""
    epsilon = 1e-15  # Prevent log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Categorical Cross-Entropy (simplified)
def categorical_cross_entropy(y_true, y_pred):
    """Categorical Cross-Entropy Loss"""
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred))

# Example: Binary classification
y_true_binary = np.array([1, 0, 1, 0])
y_pred_binary = np.array([0.9, 0.1, 0.8, 0.2])

# Example: Multi-class classification
y_true_multi = np.array([0, 0, 1, 0])  # One-hot encoding for class 2
y_pred_multi = np.array([0.1, 0.2, 0.6, 0.1])

print("Classification Loss Functions:")
print(f"Binary Cross-Entropy: {binary_cross_entropy(y_true_binary, y_pred_binary):.4f}")
print(f"Categorical Cross-Entropy: {categorical_cross_entropy(y_true_multi, y_pred_multi):.4f}")

# Visualize binary cross-entropy
p = np.linspace(0.01, 0.99, 100)
loss_when_true = -np.log(p)
loss_when_false = -np.log(1 - p)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(p, loss_when_true, 'b-', label='y_true=1', linewidth=2)
plt.plot(p, loss_when_false, 'r-', label='y_true=0', linewidth=2)
plt.xlabel('Predicted Probability')
plt.ylabel('Loss')
plt.title('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Confusion matrix-like visualization
classes = ['Class 0', 'Class 1']
predictions = [y_pred_binary[0], y_pred_binary[1]]
true_labels = ['Class 1', 'Class 0']
colors = ['green' if abs(p - t) < 0.2 else 'red' for p, t in zip(y_pred_binary, y_true_binary)]

plt.bar(classes, y_pred_binary, color=colors, alpha=0.6)
plt.axhline(y=0.5, color='red', linestyle='--', label='Threshold=0.5')
plt.title('Binary Classification Predictions')
plt.ylabel('Predicted Probability')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 ML Connection:")
print("Cross-entropy is widely used in classification tasks")
print("Encourages models to make confident predictions")

### 2. Gradient Descent Variant 

#### 2.1 Batch Gradient Descent

Batch GD uses the entire dataset to compute gradients. It's accurate but slow for large datasets.

In [ ]:
class BatchGradientDescent:
    def __init__(self, learning_rate=0.01, n_iterations=100):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for iteration in range(self.n_iterations):
            # Forward pass
            y_pred = np.dot(X, self.weights) + self.bias
            
            # Compute loss
            loss = np.mean((y - y_pred) ** 2)
            self.loss_history.append(loss)
            
            # Compute gradients (using entire dataset)
            dw = -2 * np.dot(X.T, (y - y_pred)) / n_samples
            db = -2 * np.mean(y - y_pred)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Loss = {loss:.4f}")
        
        return self

# Generate sample data
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1).flatten()

# Train using Batch GD
model = BatchGradientDescent(learning_rate=0.01, n_iterations=100)
model.fit(X, y)

print(f"\nFinal parameters:")
print(f"Weight: {model.weights[0]:.4f}")
print(f"Bias: {model.bias:.4f}")

# Visualize training
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(model.loss_history, 'b-', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Batch GD: Training Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.6, label='Data')
X_line = np.linspace(0, 2, 100)
y_line = model.weights[0] * X_line + model.bias
plt.plot(X_line, y_line, 'r-', linewidth=2, label='Model')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Batch GD: Learned Model')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 ML Connection:")
print("Batch GD is deterministic and stable but computationally expensive")

### 2.2 Stochastic Gradient Descent (SGD)

SGD updates weights using one sample at a time, making it fast but noisy.

In [ ]:

class StochasticGradientDescent:
    def __init__(self, learning_rate=0.01, n_iterations=100):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for iteration in range(self.n_iterations):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            epoch_loss = 0
            
            # Update using each sample individually
            for i in range(n_samples):
                xi = X_shuffled[i:i+1]
                yi = y_shuffled[i:i+1]
                
                # Forward pass
                y_pred = np.dot(xi, self.weights) + self.bias
                
                # Compute gradient (single sample)
                dw = -2 * xi.T * (yi - y_pred)
                db = -2 * (yi - y_pred)
                
                # Update parameters
                self.weights -= self.lr * dw.flatten()
                self.bias -= self.lr * db.flatten()
                
                epoch_loss += (yi - y_pred) ** 2
            
            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss[0])
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Loss = {avg_loss[0]:.4f}")
        
        return self

# Train using SGD
model_sgd = StochasticGradientDescent(learning_rate=0.01, n_iterations=100)
model_sgd.fit(X, y)

print(f"\nFinal parameters (SGD):")
print(f"Weight: {model_sgd.weights[0]:.4f}")
print(f"Bias: {model_sgd.bias:.4f}")

# Compare SGD vs Batch GD
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(model.loss_history, 'b-', label='Batch GD', alpha=0.7, linewidth=2)
plt.plot(model_sgd.loss_history, 'r-', label='SGD', alpha=0.7, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.6, label='Data')
X_line = np.linspace(0, 2, 100)
y_line_sgd = model_sgd.weights[0] * X_line + model_sgd.bias
plt.plot(X_line, y_line_sgd, 'r-', linewidth=2, label='SGD Model')
plt.xlabel('X')
plt.ylabel('y')
plt.title('SGD: Learned Model')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 ML Connection:")
print("SGD is faster but noisier than Batch GD")
print("Noise can help escape local minima!")

### 2.3 Mini-Batch Gradient Descent

Mini-Batch GD uses small batches of data, balancing speed and stability.

In [ ]:
class MiniBatchGradientDescent:
    def __init__(self, learning_rate=0.01, batch_size=20, n_iterations=100):
        self.lr = learning_rate
        self.batch_size = batch_size
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for iteration in range(self.n_iterations):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # Mini-batch training
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]
                
                # Forward pass
                y_pred = np.dot(X_batch, self.weights) + self.bias
                
                # Compute gradients (batch)
                dw = -2 * np.dot(X_batch.T, (y_batch - y_pred)) / len(X_batch)
                db = -2 * np.mean(y_batch - y_pred)
                
                # Update parameters
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
            
            # Compute full loss
            y_pred_all = np.dot(X, self.weights) + self.bias
            loss = np.mean((y - y_pred_all) ** 2)
            self.loss_history.append(loss)
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Loss = {loss:.4f}")
        
        return self

# Compare all three variants
batch_sizes = ['Full (Batch)', 'Mini-Batch (20)', 'SGD (1)']
colors = ['blue', 'green', 'red']

plt.figure(figsize=(12, 6))

# Batch GD
model_batch = BatchGradientDescent(learning_rate=0.01, n_iterations=100)
model_batch.fit(X, y)

# Mini-Batch GD
model_mini = MiniBatchGradientDescent(learning_rate=0.01, batch_size=20, n_iterations=100)
model_mini.fit(X, y)

# SGD
model_sgd = StochasticGradientDescent(learning_rate=0.01, n_iterations=100)
model_sgd.fit(X, y)

plt.plot(model_batch.loss_history, 'b-', label='Batch GD', linewidth=2)
plt.plot(model_mini.loss_history, 'g-', label='Mini-Batch GD', linewidth=2)
plt.plot(model_sgd.loss_history, 'r-', label='SGD', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Gradient Descent Variants Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

print("\n💡 ML Connection:")
print("Mini-Batch GD is the most popular choice in deep learning")
print("It balances speed (SGD) and stability (Batch GD)")

### 3. LEARNING RATE EFFECTS

####  3.1 Impact of Learning Rate

Learning rate determines step size in gradient descent. Too large causes divergence; too small slows convergence.

In [ ]:
# Test different learning rates
def test_learning_rates():
    learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
    colors = ['blue', 'green', 'orange', 'red', 'purple']
    
    plt.figure(figsize=(12, 6))
    
    # Use a simple quadratic function
    def gradient_descent_lr(X, y, lr, iterations=50):
        w = np.random.randn(1)
        losses = []
        
        for _ in range(iterations):
            y_pred = w * X
            loss = np.mean((y - y_pred) ** 2)
            losses.append(loss)
            
            # Gradient
            dw = -2 * np.mean(X * (y - y_pred))
            w -= lr * dw
        
        return losses
    
    # Generate data
    np.random.seed(42)
    X_lr = np.linspace(0, 2, 100).reshape(-1, 1)
    y_lr = 3 * X_lr.flatten() + np.random.randn(100) * 0.5
    
    for lr, color in zip(learning_rates, colors):
        losses = gradient_descent_lr(X_lr, y_lr, lr)
        plt.plot(losses, color=color, label=f'LR = {lr}', linewidth=2)
    
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('Effect of Learning Rate on Convergence')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 10)
    plt.show()

test_learning_rates()

print("\n💡 ML Connection:")
print("Learning rate is a critical hyperparameter")
print("Too high → Model diverges")
print("Too low → Training is very slow")
print("Optimal → Stable and fast learning")

###  4. Advanced Optimizers 

#### 4.1 Momentum

Momentum accelerates gradient descent by accumulating past gradients.


In [ ]:
class MomentumOptimizer:
    def __init__(self, learning_rate=0.01, momentum=0.9):
        self.lr = learning_rate
        self.momentum = momentum
        self.velocity_w = 0
        self.velocity_b = 0
    
    def update(self, weights, bias, grad_w, grad_b):
        # Update velocities
        self.velocity_w = self.momentum * self.velocity_w - self.lr * grad_w
        self.velocity_b = self.momentum * self.velocity_b - self.lr * grad_b
        
        # Update parameters
        weights += self.velocity_w
        bias += self.velocity_b
        
        return weights, bias

# Compare SGD vs Momentum
def compare_sgd_momentum():
    np.random.seed(42)
    X_mom = 2 * np.random.rand(100, 1)
    y_mom = 4 + 3 * X_mom.flatten() + np.random.randn(100) * 0.5
    
    # SGD
    sgd = StochasticGradientDescent(learning_rate=0.01, n_iterations=50)
    sgd.fit(X_mom, y_mom)
    
    # With Momentum (modified SGD)
    class SGDWithMomentum:
        def __init__(self, learning_rate=0.01, momentum=0.9):
            self.lr = learning_rate
            self.momentum = momentum
            self.weights = None
            self.bias = None
            self.loss_history = []
            self.velocity_w = 0
            self.velocity_b = 0
        
        def fit(self, X, y):
            n_samples, n_features = X.shape
            self.weights = np.zeros(n_features)
            self.bias = 0
            
            for iteration in range(50):
                # Shuffle data
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                y_shuffled = y[indices]
                
                epoch_loss = 0
                
                for i in range(n_samples):
                    xi = X_shuffled[i:i+1]
                    yi = y_shuffled[i:i+1]
                    
                    # Forward pass
                    y_pred = np.dot(xi, self.weights) + self.bias
                    
                    # Gradient
                    dw = -2 * xi.T * (yi - y_pred)
                    db = -2 * (yi - y_pred)
                    
                    # Update with momentum
                    self.velocity_w = self.momentum * self.velocity_w - self.lr * dw.flatten()
                    self.velocity_b = self.momentum * self.velocity_b - self.lr * db.flatten()
                    
                    self.weights += self.velocity_w
                    self.bias += self.velocity_b
                    
                    epoch_loss += (yi - y_pred) ** 2
                
                avg_loss = epoch_loss / n_samples
                self.loss_history.append(avg_loss[0])
            
            return self
    
    sgd_momentum = SGDWithMomentum(learning_rate=0.01, momentum=0.9)
    sgd_momentum.fit(X_mom, y_mom)
    
    # Plot comparison
    plt.figure(figsize=(10, 5))
    plt.plot(sgd.loss_history, 'b-', label='SGD', linewidth=2)
    plt.plot(sgd_momentum.loss_history, 'r-', label='SGD + Momentum', linewidth=2)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('SGD vs SGD with Momentum')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\n💡 ML Connection:")
    print("Momentum helps escape local minima")
    print("Accelerates convergence in areas of low gradient")

compare_sgd_momentum()

### 4.2 Adam Optimizer

Adam combines momentum and adaptive learning rates for efficient optimization.

In [ ]:
class AdamOptimizerDetailed:
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m_w = 0
        self.v_w = 0
        self.m_b = 0
        self.v_b = 0
        self.t = 0
        
    def update(self, weights, bias, grad_w, grad_b):
        self.t += 1
        
        # Update moments for weights
        self.m_w = self.beta1 * self.m_w + (1 - self.beta1) * grad_w
        self.v_w = self.beta2 * self.v_w + (1 - self.beta2) * (grad_w ** 2)
        
        # Update moments for bias
        self.m_b = self.beta1 * self.m_b + (1 - self.beta1) * grad_b
        self.v_b = self.beta2 * self.v_b + (1 - self.beta2) * (grad_b ** 2)
        
        # Bias correction
        m_w_hat = self.m_w / (1 - self.beta1 ** self.t)
        v_w_hat = self.v_w / (1 - self.beta2 ** self.t)
        m_b_hat = self.m_b / (1 - self.beta1 ** self.t)
        v_b_hat = self.v_b / (1 - self.beta2 ** self.t)
        
        # Update parameters
        weights -= self.lr * m_w_hat / (np.sqrt(v_w_hat) + self.epsilon)
        bias -= self.lr * m_b_hat / (np.sqrt(v_b_hat) + self.epsilon)
        
        return weights, bias

# Compare optimizers on a difficult function
def compare_optimizers_difficult():
    np.random.seed(42)
    X_opt = 2 * np.random.rand(200, 1)
    y_opt = np.sin(3 * X_opt.flatten()) + np.random.randn(200) * 0.1
    
    class NeuralNetwork:
        def __init__(self):
            self.w1 = np.random.randn(1)
            self.w2 = np.random.randn(1)
            self.b = np.random.randn(1)
        
        def forward(self, X):
            return self.w1 * X + self.w2 * X**2 + self.b
        
        def loss(self, X, y):
            pred = self.forward(X)
            return np.mean((y - pred) ** 2)
        
        def gradients(self, X, y):
            pred = self.forward(X)
            grad_w1 = -2 * np.mean(X * (y - pred))
            grad_w2 = -2 * np.mean(X**2 * (y - pred))
            grad_b = -2 * np.mean(y - pred)
            return grad_w1, grad_w2, grad_b
    
    # Train with different optimizers
    def train_with_optimizer(optimizer_class, **kwargs):
        model = NeuralNetwork()
        opt = optimizer_class(**kwargs)
        losses = []
        
        for _ in range(100):
            grad_w1, grad_w2, grad_b = model.gradients(X_opt, y_opt)
            
            # Update parameters
            model.w1, model.b = opt.update(model.w1, model.b, grad_w1, grad_b)
            # For simplicity, update w2 separately
            model.w2 -= 0.001 * grad_w2
            
            losses.append(model.loss(X_opt, y_opt))
        
        return losses
    
    # Compare
    losses_sgd = train_with_optimizer(lambda **kw: type('Opt', (), {'update': lambda self, w, b, gw, gb: (w - 0.01*gw, b - 0.01*gb)})())
    
    # Simplified Adam
    class SimpleAdam:
        def __init__(self):
            self.m_w = 0
            self.v_w = 0
            self.m_b = 0
            self.v_b = 0
            self.t = 0
        
        def update(self, w, b, gw, gb):
            self.t += 1
            beta1, beta2 = 0.9, 0.999
            self.m_w = beta1 * self.m_w + 0.1 * gw
            self.v_w = beta2 * self.v_w + 0.001 * (gw ** 2)
            self.m_b = beta1 * self.m_b + 0.1 * gb
            self.v_b = beta2 * self.v_b + 0.001 * (gb ** 2)
            
            m_w_hat = self.m_w / (1 - beta1 ** self.t)
            v_w_hat = self.v_w / (1 - beta2 ** self.t)
            m_b_hat = self.m_b / (1 - beta1 ** self.t)
            v_b_hat = self.v_b / (1 - beta2 ** self.t)
            
            w -= 0.01 * m_w_hat / (np.sqrt(v_w_hat) + 1e-8)
            b -= 0.01 * m_b_hat / (np.sqrt(v_b_hat) + 1e-8)
            
            return w, b
    
    losses_adam = train_with_optimizer(SimpleAdam)
    
    plt.figure(figsize=(10, 5))
    plt.plot(losses_sgd, 'b-', label='SGD', linewidth=2)
    plt.plot(losses_adam, 'r-', label='Adam', linewidth=2)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('SGD vs Adam on Complex Function')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.show()
    
    print("\n💡 ML Connection:")
    print("Adam is the most popular optimizer in deep learning")
    print("Used in GPT, BERT, Stable Diffusion, and more!")

compare_optimizers_difficult()

### 5. Optimization Visualization

#### 5.1 Optimization Landscape Visualization


Visualizing the optimization landscape helps understand how different algorithms navigate towards minima.

In [ ]:



# Create a complex optimization landscape
def complex_function(x, y):
    """Rosenbrock-like function with multiple minima"""
    return (1 - x)**2 + 100 * (y - x**2)**2 + 0.5 * np.sin(3 * x) * np.cos(5 * y)

# Visualize the landscape
x = np.linspace(-1.5, 1.5, 30)
y = np.linspace(-1, 2, 30)
X, Y = np.meshgrid(x, y)
Z = complex_function(X, Y)

fig = plt.figure(figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x,y)')
ax1.set_title('Optimization Landscape')

# Contour with possible paths
ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=30, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Contour Map (Goal: Reach minimum)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 ML Connection:")
print("The goal of optimization is to find the global minimum")
print("Complex landscapes like this exist in neural networks!")

### 6. Quick Reference Summary

#### 6.1 Optimization Cheatsheet

In [ ]:

# Optimization quick reference
print("="*60)
print("⚙️ OPTIMIZATION CHEATSHEET")
print("="*60)

optimization_concepts = {
    "Loss Function": "Measures model performance",
    "Gradient Descent": "Iterative optimization algorithm",
    "Batch GD": "Uses entire dataset",
    "SGD": "Uses single sample",
    "Mini-Batch GD": "Uses small batches",
    "Learning Rate": "Controls step size",
    "Momentum": "Accumulates past gradients",
    "Adam": "Adaptive momentum optimizer",
    "Convergence": "Reaching optimal solution",
    "Global Minimum": "Best possible solution"
}

for concept, description in optimization_concepts.items():
    print(f"📌 {concept}: {description}")

